In [12]:
import pandas as pd
import geopandas as gpd
import requests
import time
from tqdm import tqdm
tqdm.pandas()

In [13]:
facility_file = 'C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/서울시 빗물관리시설 통계.csv'
pump_file = 'C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/서울시 빗물펌프장 공간정보.csv'

In [30]:
# 1. 빗물관리시설 방어력 (AC1) 전처리
try:
    facility_df = pd.read_csv(facility_file, encoding='cp949')
except:
    facility_df = pd.read_csv(facility_file, encoding='utf-8')

# 결측치를 0으로 채우고 자치구별로 총합(sum) 계산
facility_df['저류 용량(㎥)'] = facility_df['저류 용량(㎥)'].fillna(0)
facility_df['빗물 관리량(㎥/h)'] = facility_df['빗물 관리량(㎥/h)'].fillna(0)

ac1_df = facility_df.groupby('자치구명')[['저류 용량(㎥)', '빗물 관리량(㎥/h)']].sum().reset_index()
ac1_df.rename(columns={'자치구명': '자치구'}, inplace=True)

In [31]:
# 2. 빗물펌프장 인프라 (AC2) 전처리
try:
    pump_df = pd.read_csv(pump_file, encoding='cp949')
except:
    pump_df = pd.read_csv(pump_file, encoding='utf-8')

# '기타' 컬럼의 숫자가 자치구 고유 코드를 포함함 (예: 1203 -> 12)
# 100으로 나눈 몫을 취해 앞자리 지역코드만 추출
pump_df['구코드'] = pump_df['기타'] // 100

In [32]:
# 파악된 구코드 -> 자치구명 역매핑 딕셔너리
gu_dict = {
    1: '강남구', 2: '강동구', 4: '노원구', 5: '강북구', 6: '성북구',
    7: '중랑구', 8: '동대문구', 9: '성동구', 12: '광진구', 13: '은평구',
    15: '마포구', 16: '용산구', 17: '강서구', 18: '양천구', 19: '영등포구',
    20: '구로구', 21: '동작구', 22: '금천구', 23: '관악구', 24: '서초구', 25: '송파구'
}
pump_df['자치구'] = pump_df['구코드'].map(gu_dict)

In [33]:
# 자치구별 펌프장 개수 집계
ac2_df = pump_df.groupby('자치구').size().reset_index(name='펌프장_개수')

In [34]:
# 3. 데이터 병합 및 정규화 (최종 AC 지수 산출)
ac_df = pd.merge(ac1_df, ac2_df, on='자치구', how='outer')
ac_df['펌프장_개수'] = ac_df['펌프장_개수'].fillna(0) # 펌프장이 없는 구는 0 처리

In [35]:
# 정규화 함수 (값이 클수록 방어력이 높으므로 일반 Min-Max 사용)
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())

ac_df['저류용량_정규화'] = min_max_normalize(ac_df['저류 용량(㎥)'])
ac_df['관리량_정규화'] = min_max_normalize(ac_df['빗물 관리량(㎥/h)'])
ac_df['펌프장_정규화'] = min_max_normalize(ac_df['펌프장_개수'])

In [36]:
# 세 지표의 평균을 내어 최종 적응능력 지수(AC) 산출
ac_df['AC_최종지수'] = (ac_df['저류용량_정규화'] + ac_df['관리량_정규화'] + ac_df['펌프장_정규화']) / 3

In [38]:
# 4. 결과 확인
ac_sorted = ac_df.sort_values(by='AC_최종지수', ascending=False).reset_index(drop=True)
print("\n[자치구별 적응능력(AC) 지수 산출 결과]")
display(ac_sorted[['자치구', '저류 용량(㎥)', '펌프장_개수', 'AC_최종지수']])


[자치구별 적응능력(AC) 지수 산출 결과]


,자치구,저류 용량(㎥),펌프장_개수,AC_최종지수
0,성북구,12194.0,2.0,0.711111
1,강서구,2351.0,7.0,0.373555
2,동대문구,4.0,15.0,0.359311
3,성동구,188.0,9.0,0.354181
4,송파구,14.0,6.0,0.350845
5,은평구,6740.0,1.0,0.322402
6,서초구,2203.0,7.0,0.296195
7,종로구,7326.0,0.0,0.275185
8,마포구,542.0,10.0,0.273226
9,구로구,0.0,10.0,0.264442


### 1. 빗물관리시설 방어력 지표 산출 (AC1)
- 10분 단위 우량 폭증에 대비하여 하수관거의 부담을 덜어주는 인프라 용량 집계
    - 저류 및 관리 용량 합산 : 자치구별 빗물관리시설의 빗물 관리량과 저류 용량 총합을 계산하여 1차 방어력 산출
    - 결측치 : 비어있는 곳은 0으로 채움

### 2. 위치 기반 매핑 및 펌프장 인프라 집계 (AC2)
- 강제 배출을 통해 침수를 막아내는 인프라 밀집도 산출
    - 펌프장 데이터 내 고유 관리코드를 추적하여 각 펌프장이 속한 자치구 매핑
    - 자치구별 펌프장 개수 : 매핑된 자치구를 기준으로 펌프장의 총 개수 집계

### 3. 취약성 평가 지수로의 통합
- 산출된 시설 용량과 펌프장 개수를 정규화 후 평균하여 최종 AC 도출
    - 홍수 취약성 공식 (V = E X S - AC)에서 위험을 상쇄시키는 지표로 활용